In [1]:
import cadquery as cq

# Create a waveguide horn antenna
horn = (cq.Workplane("XY")
    .box(20, 10, 30)
    .faces(">Z").workplane()
    .box(40, 20, 20, combine='cut'))

In [2]:
import FreeCAD
import Part

box = Part.makeBox(10, 10, 10)
sphere = Part.makeSphere(7)
result = box.cut(sphere)

ModuleNotFoundError: No module named 'FreeCAD'

In [3]:
import gmsh

gmsh.initialize()
gmsh.model.add("waveguide")

# Create geometry
rect = gmsh.model.occ.addRectangle(0, 0, 0, 10, 5)
gmsh.model.occ.synchronize()

# Define physical groups for EM simulation
gmsh.model.addPhysicalGroup(2, [rect], 1)
gmsh.model.setPhysicalName(2, 1, "waveguide_cross_section")

gmsh.model.mesh.generate(2)
gmsh.write("waveguide.msh")
gmsh.finalize()

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.000625745s, CPU 0.000963s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.0091858s, CPU 0.008483s)
Info    : 70 nodes 142 elements
Info    : Writing 'waveguide.msh'...
Info    : Done writing 'waveguide.msh'


In [ ]:
import salome
salome.salome_init()

from salome.geom import geomBuilder
geom = geomBuilder.New()

# Import CAD and perform healing
antenna_assembly = geom.ImportSTEP("antenna_housing.step", True, True)
healed = geom.RemoveExtraEdges(antenna_assembly, True)

# Create mesh with different element sizes for different regions
from salome.smesh import smeshBuilder
mesh = smeshBuilder.New()
antenna_mesh = mesh.Mesh(healed)

# Define algorithms and parameters
mesh.Triangle().MaxElementArea(1.0)
mesh.AutomaticHexahedralization()
mesh.Compute()

# Export with named groups
mesh.ExportMED("antenna_mesh.med")

In [ ]:
from dolfinx import fem, mesh
# Define Maxwell's equations weak form almost as you'd write it on paper
a = inner(curl(E), curl(v)) * dx - k0**2 * inner(eps * E, v) * dx

In [4]:
import pyvista as pv

mesh = pv.read('results.vtu')
mesh.plot(scalars='temperature', cmap='hot')

: 